In [2]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import torch

In [3]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix

# load data
X, y = [], []
for file in os.listdir("../preprocessed_data"):
    if file.endswith(".npy"):
        label = int(file.split("_")[3])  # from e.g. ..._label_0_stripped.npy
        vol = np.load(os.path.join("../preprocessed_data", file))  # shape: (1, D, H, W)
        X.append(vol)
        y.append(label)

# horizontal flip along width axis (W = axis 3)
X_aug = []
y_aug = []
for i in range(len(X)):
    flipped = np.flip(X[i], axis=3)  # width flip
    X_aug.append(flipped)
    y_aug.append(y[i])

X = np.concatenate([X, np.array(X_aug)])
y = np.concatenate([y, np.array(y_aug)])

# flip vertically (height axis)
X_flip_v = [np.flip(x, axis=2) for x in X]
y_flip_v = y.copy()

# add Gaussian noise
X_noise = [x + np.random.normal(0, 0.01, x.shape) for x in X]
y_noise = y.copy()

# merge all
X = np.concatenate([X, np.array(X_flip_v), np.array(X_noise)])
y = np.concatenate([y, np.array(y_flip_v), np.array(y_noise)])

X = np.array(X)
y = np.array(y)

# normalize
X = (X - np.mean(X)) / np.std(X)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# sample weights
class_sample_count = np.array([np.sum(y_train == t) for t in np.unique(y_train)])
weights = 1. / class_sample_count
sample_weights = weights[y_train]  # assign weight to each sample
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


# dataloader
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=2,
    sampler=sampler  # no shuffle needed
)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=2)

# 3D CNN (small)
class Improved3DCNN(nn.Module):
    def __init__(self):
        super(Improved3DCNN, self).__init__()
        self.conv1 = nn.Conv3d(1, 4, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(4)
        self.conv2 = nn.Conv3d(4, 8, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(8)
        self.conv3 = nn.Conv3d(8, 16, 3, padding=1)
        self.bn3 = nn.BatchNorm3d(16)
        self.pool = nn.MaxPool3d(2)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(16 * 8 * 16 * 16, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)

# setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Improved3DCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# train
for epoch in range(10):
    model.train()
    total_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1} | Loss: {total_loss / len(train_loader):.4f}")

# test

model.eval()
correct = 0
total = 0
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(predicted.cpu().numpy())

# accuracy & confusion matrix
y_true = np.array(y_true)
y_pred = np.array(y_pred)
accuracy = np.mean(y_pred == y_true)

print(f"Test Accuracy: {accuracy:.2%}")
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


Epoch 1 | Loss: 0.7514
Epoch 2 | Loss: 0.5364
Epoch 3 | Loss: 0.4125
Epoch 4 | Loss: 0.4536
Epoch 5 | Loss: 0.3416
Epoch 6 | Loss: 0.3322
Epoch 7 | Loss: 0.3093
Epoch 8 | Loss: 0.2536
Epoch 9 | Loss: 0.2422
Epoch 10 | Loss: 0.2108
Test Accuracy: 97.18%
Confusion matrix:
 [[22  0]
 [ 2 47]]


In [4]:
# ratio:2 : 3 : 5 to three client
num_total = len(X)
indices = np.arange(num_total)
np.random.shuffle(indices)

split1 = int(0.2 * num_total)
split2 = split1 + int(0.3 * num_total)

client_indices_list = [
    indices[:split1],
    indices[split1:split2],
    indices[split2:],
]

client_data = []

for i, client_indices in enumerate(client_indices_list):
    client_X = X[client_indices]
    client_y = y[client_indices]

    # 80% training and 20% testing
    X_train, X_test, y_train, y_test = train_test_split(
        client_X, client_y, test_size=0.2, random_state=42)

    # standardization
    for j in range(len(X_train)):
        X_train[j] = (X_train[j] - np.mean(X_train[j])) / (np.std(X_train[j]) + 1e-8)
    for j in range(len(X_test)):
        X_test[j] = (X_test[j] - np.mean(X_test[j])) / (np.std(X_test[j]) + 1e-8)


    #transfer to tensor and DataLoader
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=2, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=2)

    client_data.append((train_loader, test_loader))



In [5]:
# Check box
for i, (train_loader, test_loader) in enumerate(client_data):
    train_samples = len(train_loader.dataset)
    test_samples = len(test_loader.dataset)
    batch = next(iter(train_loader))
    data_shape = batch[0].shape
    print(f"Client {i + 1}:")
    print(f"  Train samples: {train_samples}")
    print(f"  Test samples:  {test_samples}")
    print(f"  Sample shape:  {data_shape}")
    print("-" * 40)
print("origin:", len([f for f in os.listdir("../preprocessed_data") if f.endswith(".npy")]))
print("after data enhancement:", X.shape[0])

Client 1:
  Train samples: 56
  Test samples:  14
  Sample shape:  torch.Size([2, 1, 64, 128, 128])
----------------------------------------
Client 2:
  Train samples: 84
  Test samples:  22
  Sample shape:  torch.Size([2, 1, 64, 128, 128])
----------------------------------------
Client 3:
  Train samples: 142
  Test samples:  36
  Sample shape:  torch.Size([2, 1, 64, 128, 128])
----------------------------------------
origin: 59
after data enhancement: 354


In [8]:
import flwr as fl
import sys


class ParkinsonClient(fl.client.NumPyClient):
    def __init__(self, model, train_loader, test_loader, device):
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = device
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=1e-4)

    # Convert the model weights into a NumPy array and send it
    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = {k: torch.tensor(v) for k, v in params_dict}
        self.model.load_state_dict(state_dict, strict=True)

    # Training with local data
    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()
        for inputs, labels in self.train_loader:
            inputs, labels = inputs.to(self.device), labels.to(self.device)
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
        return self.get_parameters({}), len(self.train_loader.dataset), {}

    # Returns the loss value, number of samples, accuracy.
    def evaluate(self, parameters, config):
        print(f"[Client] Number of test samples: {len(self.test_loader.dataset)}", file=sys.stdout, flush=True)
        self.set_parameters(parameters)
        self.model.eval()
        correct, total, loss = 0, 0, 0.0
        with torch.no_grad():
            for inputs, labels in self.test_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                outputs = self.model(inputs)
                loss += self.criterion(outputs, labels).item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        accuracy = correct / total
        print(f"[Client] Eval Accuracy: {accuracy:.2%}, Loss: {loss:.4f}", file=sys.stdout, flush=True)
        return loss, total, {"accuracy": accuracy}



In [9]:
import threading

for i, (train_loader, test_loader) in enumerate(client_data):
    model = Improved3DCNN().to(device)
    client = ParkinsonClient(model, train_loader, test_loader, device)

    threading.Thread(
    target=fl.client.start_numpy_client,
    kwargs={
        "server_address": "localhost:8080",
        "client": client
    }
).start()

	Instead, use `flwr.client.start_client()` by ensuring you first call the `.to_client()` method as shown below: 
	flwr.client.start_client(
		server_address='<IP>:<PORT>',
		client=FlowerClient().to_client(), # <-- where FlowerClient is of type flwr.client.NumPyClient object
	)
	Using `start_numpy_client()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
	Instead, use the `flower-supernode` CLI command to start a SuperNode as shown below:

		$ flower-supernode --insecure --superlink='<IP>:<PORT>'

	To view all available options, run:

		$ flower-supernode --help

	Using `start_client()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
	Instead, use `flwr.client.start_client()` by ensuring you first call the `.to_client()` method as shown below: 
	flwr.client.start_client(
		server_address='<IP>:<PORT>',
		client